# Feature Engineering
**Input:** `data/final/events_final.csv`

**Outputs** (en `data/final/`):
- `user_clustering_dataset.csv` — matriz numérica normalizada para KMeans y GMM
- `interacciones_ponderadas.csv` — interacciones usuario-producto ponderadas por tipo de evento (input para SVD)

In [1]:
import os
import pandas as pd
import numpy as np

OUTPUT_DIR = 'data/final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ruta robusta de entrada para evitar errores por cwd del notebook
candidate_inputs = [
    'inferido_limpio.csv',
    'data/final/events_final.csv',
    '/Users/nicolasdiaz/Desktop/pf/inferido_limpio.csv',
    '/Users/nicolasdiaz/Desktop/pf/data/final/events_final.csv',
]
input_path = next((p for p in candidate_inputs if os.path.exists(p)), None)
if input_path is None:
    raise FileNotFoundError('No se encontró inferido_limpio.csv ni events_final.csv en rutas esperadas.')

df = pd.read_csv(input_path, parse_dates=['event_time'])
df['event_time'] = df['event_time'].dt.tz_localize(None)

# Extraer hora (se usa en features de usuario)
df['hora'] = df['event_time'].dt.hour

print(f'Input usado: {input_path}')
print(df.shape)
df.head(3)

FileNotFoundError: No se encontró inferido_limpio.csv ni events_final.csv en rutas esperadas.

In [ ]:
# ------------------------------------------------------------
# 1. Peso por tipo de evento
# El EDA confirmó que el 89% son views y solo el 4% compras.

In [10]:
# purchase = 3, cart = 2, view = 1
peso_evento = {'purchase': 3, 'cart': 2, 'view': 1}
df['peso'] = df['event_type'].map(peso_evento)

print('Pesos asignados:')
print(df.groupby('event_type')['peso'].first())

Pesos asignados:
event_type
cart        2
purchase    3
view        1
Name: peso, dtype: int64


In [ ]:
# Fallback por si la celda se ejecuta fuera de orden
if 'peso' not in df.columns:
    peso_evento = {'purchase': 3, 'cart': 2, 'view': 1}
    df['peso'] = df['event_type'].map(peso_evento)

df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df['event_weight'] = df['event_type'].map({'view': 1, 'cart': 2, 'purchase': 3})

user_base = df.groupby('user_id').agg(
    total_events=('event_type', 'count'),
    unique_products=('product_id', 'nunique'),
    avg_price=('price', 'mean'),
    weighted_score=('event_weight', 'sum'),
    last_event=('event_time', 'max')
).reset_index()

event_counts = df.pivot_table(
    index='user_id',
    columns='event_type',
    values='product_id',
    aggfunc='count',
    fill_value=0
).reset_index()

for col in ['view', 'cart', 'purchase']:
    if col not in event_counts.columns:
        event_counts[col] = 0

user_df = user_base.merge(event_counts, on='user_id', how='left')

user_df['avg_events_per_product'] = np.where(
    user_df['unique_products'] > 0,
    user_df['total_events'] / user_df['unique_products'],
    0
)

purchase_df = df[df['event_type'] == 'purchase']
avg_purchase = purchase_df.groupby('user_id').agg(
    avg_purchase_price=('price', 'mean')
).reset_index()

user_df = user_df.merge(avg_purchase, on='user_id', how='left')
user_df['avg_purchase_price'] = user_df['avg_purchase_price'].fillna(0)

dataset_end_date = df['event_time'].max()
user_df['recency_days'] = (dataset_end_date - user_df['last_event']).dt.days
user_df = user_df.drop(columns=['last_event'])

df['hour'] = df['event_time'].dt.hour
conditions = [
    (df['hour'] >= 0) & (df['hour'] < 6),
    (df['hour'] >= 6) & (df['hour'] < 12),
    (df['hour'] >= 12) & (df['hour'] < 18),
    (df['hour'] >= 18) & (df['hour'] < 24)
]
choices = ['night', 'morning', 'afternoon', 'evening']
df['time_period'] = np.select(conditions, choices, default='unknown')

time_counts = df.pivot_table(
    index='user_id',
    columns='time_period',
    values='product_id',
    aggfunc='count',
    fill_value=0
)

for col in ['morning', 'afternoon', 'evening', 'night']:
    if col not in time_counts.columns:
        time_counts[col] = 0

time_props = time_counts.div(time_counts.sum(axis=1), axis=0)
time_props = time_props.rename(columns={
    'morning': 'pct_morning',
    'afternoon': 'pct_afternoon',
    'evening': 'pct_evening',
    'night': 'pct_night'
}).reset_index()

user_df = user_df.merge(
    time_props[['user_id', 'pct_morning', 'pct_afternoon', 'pct_evening', 'pct_night']],
    on='user_id',
    how='left'
)

df['dayofweek'] = df['event_time'].dt.dayofweek
df['week_period'] = np.where(df['dayofweek'] < 5, 'weekday', 'weekend')

week_counts = df.pivot_table(
    index='user_id',
    columns='week_period',
    values='product_id',
    aggfunc='count',
    fill_value=0
)

for col in ['weekday', 'weekend']:
    if col not in week_counts.columns:
        week_counts[col] = 0

week_props = week_counts.div(week_counts.sum(axis=1), axis=0)
week_props = week_props.rename(columns={
    'weekday': 'pct_weekday',
    'weekend': 'pct_weekend'
}).reset_index()

user_df = user_df.merge(
    week_props[['user_id', 'pct_weekday', 'pct_weekend']],
    on='user_id',
    how='left'
)

category_mapping = {
    'electronics': 'tech',
    'computers': 'tech',
    'auto': 'tech',
    'apparel': 'fashion_lifestyle',
    'accessories': 'fashion_lifestyle',
    'jewelry': 'fashion_lifestyle',
    'kids': 'fashion_lifestyle',
    'furniture': 'home',
    'appliances': 'home',
    'construction': 'home',
    'country_yard': 'home',
    'stationery': 'home',
    'sport': 'welfare',
    'medicine': 'welfare'
}

df['macro_category'] = df['nivel1'].map(category_mapping)

category_counts = df.pivot_table(
    index='user_id',
    columns='macro_category',
    values='product_id',
    aggfunc='count',
    fill_value=0
)

for col in ['tech', 'fashion_lifestyle', 'home', 'welfare']:
    if col not in category_counts.columns:
        category_counts[col] = 0

category_props = category_counts.div(category_counts.sum(axis=1), axis=0)
category_props = category_props.rename(columns={
    'tech': 'proportion_tech',
    'fashion_lifestyle': 'proportion_fashion_lifestyle',
    'home': 'proportion_home',
    'welfare': 'proportion_welfare'
}).reset_index()

user_df = user_df.merge(category_props, on='user_id', how='left')

columnas_finales = [
    'user_id', 'total_events', 'unique_products', 'view', 'cart', 'purchase',
    'avg_price', 'avg_events_per_product', 'avg_purchase_price', 'weighted_score',
    'recency_days', 'pct_morning', 'pct_afternoon', 'pct_evening', 'pct_night',
    'pct_weekday', 'pct_weekend', 'proportion_tech', 'proportion_fashion_lifestyle',
    'proportion_home', 'proportion_welfare'
]

clustering = user_df[columnas_finales]

clustering_out = os.path.join(OUTPUT_DIR, 'user_clustering_dataset.csv')
clustering.to_csv(clustering_out, index=False, encoding='utf-8-sig')
print(f'Guardado: {clustering_out}')
print(f'Usuarios únicos: {len(clustering):,}')
print(f'Columnas: {list(clustering.columns)}')
clustering.head()

In [ ]:
# ------------------------------------------------------------
# 4. Tabla de interacciones ponderadas
# Input principal para el modelo de filtrado colaborativo.

In [ ]:
# Una fila por par usuario-producto, sumando todos los pesos de sus interacciones
interacciones = df.groupby(['user_id', 'product_id']).agg(
    score              = ('peso', 'sum'),
    n_interacciones    = ('event_type', 'count'),
    ultima_interaccion = ('event_time', 'max'),
).reset_index()

# Recencia de la interacción: días desde la última vez que el usuario tocó ese producto
fecha_referencia = df['event_time'].max()
interacciones['dias_desde_interaccion'] = (
    fecha_referencia - interacciones['ultima_interaccion']
).dt.days

# Score ajustado por recencia: interacciones recientes pesan más
interacciones['score_temporal'] = (
    interacciones['score'] * np.exp(-0.01 * interacciones['dias_desde_interaccion'])
).round(4)

interacciones.drop(columns=['ultima_interaccion'], inplace=True)

print(f'Pares usuario-producto únicos: {len(interacciones):,}')
print(f'Score máximo: {interacciones["score"].max()}')
print(f'Score temporal máximo: {interacciones["score_temporal"].max():.2f}')
interacciones.head(10)

In [ ]:
interacciones_out = os.path.join(OUTPUT_DIR, 'interacciones_ponderadas.csv')
interacciones.to_csv(interacciones_out, index=False)
print(f'Guardado: {interacciones_out}')

In [ ]:
# ------------------------------------------------------------
# Resumen de archivos generados

In [ ]:
print('=== ARCHIVOS GENERADOS ===')
print(f'user_clustering_dataset.csv  → {len(clustering):,} filas, {len(clustering.columns)} columnas')
print(f'interacciones_ponderadas.csv → {len(interacciones):,} filas, {len(interacciones.columns)} columnas')
print()
print('=== COLUMNAS ===')
print('user_clustering_dataset:', list(clustering.columns))
print('interacciones_ponderadas:', list(interacciones.columns))